# Fine-tune DistilBERT for Sentiment Analysis

This notebook trains a binary sentiment classifier and saves it to `models/sentiment-model/`.
The API, Streamlit app, and desktop GUI automatically use that local model when it exists.

In [ ]:
!pip install datasets transformers evaluate scikit-learn accelerate

In [ ]:
import json
from pathlib import Path

import evaluate
import numpy as np
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = Path("../models/sentiment-model")
METRICS_PATH = Path("../models/metrics.json")
TRAIN_SAMPLES = 4000
EVAL_SAMPLES = 1000

In [ ]:
dataset = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=128)


tokenized = dataset.map(tokenize, batched=True)
train_dataset = tokenized["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
eval_dataset = tokenized["validation"].shuffle(seed=42).select(range(EVAL_SAMPLES))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="../test_trainer",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
print("Evaluation metrics:", metrics)

predictions = trainer.predict(eval_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids
target_names = ["NEGATIVE", "POSITIVE"]

print(classification_report(true_labels, pred_labels, target_names=target_names))
print("Confusion matrix:\n", confusion_matrix(true_labels, pred_labels))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved model to {OUTPUT_DIR.resolve()}")
print(f"Saved metrics to {METRICS_PATH.resolve()}")